# KLUE-BERT 네이버 영화 리뷰 감성 분류
## 간소화 버전


In [1]:
# !pip install transformers

## 1. 라이브러리 및 데이터 로드

In [2]:
import pandas as pd
import numpy as np
import urllib.request
import tensorflow as tf
from transformers import BertTokenizerFast, TFBertForSequenceClassification, TextClassificationPipeline
from tensorflow.keras.callbacks import EarlyStopping

c:\AI\envs\ai\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
import transformers
print(transformers.__version__)
print(transformers.__file__)


4.44.2
c:\AI\envs\ai\lib\site-packages\transformers\__init__.py


In [4]:
import sys
!{sys.executable} -m pip install transformers==4.44.2


In [5]:
urllib.request.urlretrieve(
    "https://raw.githubusercontent.com/e9t/nsmc/master/ratings_train.txt",
    filename="ratings_train.txt"
)
urllib.request.urlretrieve(
    "https://raw.githubusercontent.com/e9t/nsmc/master/ratings_test.txt",
    filename="ratings_test.txt"
)

train_data = pd.read_table('ratings_train.txt')
test_data = pd.read_table('ratings_test.txt')

train_data.drop_duplicates(subset=['document'], inplace=True)
train_data = train_data.dropna(how='any')
test_data = test_data.dropna(how='any')

print('훈련 데이터:', len(train_data), '/ 테스트 데이터:', len(test_data))

훈련 데이터: 146182 / 테스트 데이터: 49997


In [6]:
# 샘플링 필요
train_data = train_data.sample(5000, random_state=42).reset_index(drop=True)
test_data = test_data.sample(1000, random_state=42).reset_index(drop=True)


## 2. 토크나이저 — BertTokenizerFast
v1의 convert_examples_to_features 함수 없이 한 줄로 전체 토크나이징

In [7]:
tokenizer = BertTokenizerFast.from_pretrained('klue/bert-base')

X_train_list = train_data['document'].tolist()
X_test_list = test_data['document'].tolist()
y_train = train_data['label'].tolist()
y_test = test_data['label'].tolist()

X_train = tokenizer(X_train_list, truncation=True, padding=True)
X_test = tokenizer(X_test_list, truncation=True, padding=True)

print('토큰:', X_train[0].tokens)
print('토큰 ID:', X_train[0].ids)
print('세그먼트 인코딩:', X_train[0].type_ids)
print('어텐션 마스크:', X_train[0].attention_mask)

토큰: ['[CLS]', '재밌', '##네', '##요', 'ㅎㅎ', '[SEP]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]']
토큰 ID: [2, 7478, 2203, 2182, 5311, 3, 0, 0, 0, 0, 0, 0, 0, 0

c:\AI\envs\ai\lib\site-packages\transformers\tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


## 3. 데이터셋 생성 및 모델 학습

In [8]:
train_dataset = tf.data.Dataset.from_tensor_slices((dict(X_train), y_train))
val_dataset = tf.data.Dataset.from_tensor_slices((dict(X_test), y_test))

In [9]:
# TFBertForSequenceClassification: v1에서 직접 구현한 CustomBertClassifier와 동일한 역할
model = TFBertForSequenceClassification.from_pretrained(
    'klue/bert-base',
    num_labels=2,
    from_pt=True
)

optimizer = tf.keras.optimizers.Adam(learning_rate=5e-5)
model.compile(
    optimizer=optimizer,
    loss=model.hf_compute_loss,  # 크로스 엔트로피 매핑
    metrics=['accuracy']
)

Some weights of the PyTorch model were not used when initializing the TF 2.0 model TFBertForSequenceClassification: ['bert.embeddings.position_ids']
- This IS expected if you are initializing TFBertForSequenceClassification from a PyTorch model trained on another task or with another architecture (e.g. initializing a TFBertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing TFBertForSequenceClassification from a PyTorch model that you expect to be exactly identical (e.g. initializing a TFBertForSequenceClassification model from a BertForSequenceClassification model).
Some weights or buffers of the TF 2.0 model TFBertForSequenceClassification were not initialized from the PyTorch model and are newly initialized: ['classifier.weight', 'classifier.bias']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [10]:
early_stopping = EarlyStopping(
    monitor='val_accuracy',
    min_delta=0.001,
    patience=2
)

model.fit(
    train_dataset.shuffle(10000).batch(32),
    epochs=2,
    batch_size=32,
    validation_data=val_dataset.shuffle(10000).batch(32),
    callbacks=[early_stopping]
)

Epoch 1/2


157/157 [==============================] - 694s 4s/step - loss: 0.3828 - accuracy: 0.8296 - val_loss: 0.3549 - val_accuracy: 0.8630
Epoch 2/2
157/157 [==============================] - 706s 4s/step - loss: 0.2084 - accuracy: 0.9154 - val_loss: 0.4432 - val_accuracy: 0.8470


 4569 스텝 전체 데이터(15만건) 불안해서 스텝 157(5000건)로 줄임
 

In [11]:
model.evaluate(val_dataset.batch(1024))

1/1 [==============================] - 41s 41s/step - loss: 0.4432 - accuracy: 0.8470


[0.44316133856773376, 0.847000002861023]

## 4. 모델 저장 및 테스트

In [12]:
model.save_pretrained('nsmc_model/bert-base')
tokenizer.save_pretrained('nsmc_model/bert-base')

('nsmc_model/bert-base\\tokenizer_config.json',
 'nsmc_model/bert-base\\special_tokens_map.json',
 'nsmc_model/bert-base\\vocab.txt',
 'nsmc_model/bert-base\\added_tokens.json',
 'nsmc_model/bert-base\\tokenizer.json')

In [13]:
loaded_tokenizer = BertTokenizerFast.from_pretrained('nsmc_model/bert-base')
loaded_model = TFBertForSequenceClassification.from_pretrained('nsmc_model/bert-base')

text_classifier = TextClassificationPipeline(
    tokenizer=loaded_tokenizer,
    model=loaded_model,
    framework='tf',
    top_k=None
)

reviews = [
    '뭐야 이 평점들은.... 나쁘진 않지만 10점짜리는 더더욱 아니잖아',
    '이 영화 존잼입니다 대박',
    '이 영화 핵노잼 ㅠㅠ',
    '와 개쩐다 정말 세계관 최강자들의 영화다',
]
for r in reviews:
    result = text_classifier(r)
    print(f'{r}\n  → {result}\n')

Some layers from the model checkpoint at nsmc_model/bert-base were not used when initializing TFBertForSequenceClassification: ['dropout_37']
- This IS expected if you are initializing TFBertForSequenceClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing TFBertForSequenceClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
All the layers of TFBertForSequenceClassification were initialized from the model checkpoint at nsmc_model/bert-base.
If your task is similar to the task the model of the checkpoint was trained on, you can already use TFBertForSequenceClassification for predictions without further training.
Hardware accelerator e.g. GPU is available in the environment, but no

뭐야 이 평점들은.... 나쁘진 않지만 10점짜리는 더더욱 아니잖아
  → [[{'label': 'LABEL_0', 'score': 0.9755093455314636}, {'label': 'LABEL_1', 'score': 0.02449066750705242}]]

이 영화 존잼입니다 대박
  → [[{'label': 'LABEL_1', 'score': 0.9957439303398132}, {'label': 'LABEL_0', 'score': 0.004256031475961208}]]

이 영화 핵노잼 ㅠㅠ
  → [[{'label': 'LABEL_0', 'score': 0.9872592091560364}, {'label': 'LABEL_1', 'score': 0.012740850448608398}]]

와 개쩐다 정말 세계관 최강자들의 영화다
  → [[{'label': 'LABEL_1', 'score': 0.9968495965003967}, {'label': 'LABEL_0', 'score': 0.0031504102516919374}]]

